# SAGARNETRA — 5-model SAR oil-spill bake-off (Colab T4)

Trains the five candidates under one harness on real Sentinel-1 tiles, scores them on the held-out test split, and exports the winner to ONNX for the API. Binary scheme (0 sea, 1 oil) — the public Kaggle masks are oil-vs-background; widen `CLASSES` later for the 5-class Krestenitis set.

**Set the runtime to GPU (T4).** You'll need a free Kaggle API token (`kaggle.json` from kaggle.com/settings).

In [ ]:
# 1. Code + training deps
!git clone https://github.com/Pjmahendra/SAGARNETRA.git
%cd SAGARNETRA
!pip install -q -r ml/requirements-train.txt kaggle

In [ ]:
# 2. Kaggle auth — upload your kaggle.json (kaggle.com/settings -> Create New API Token)
from google.colab import files
import os
files.upload()  # pick kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
# 3. Download the Sentinel-1 oil-spill dataset (~1.2 GB, binary masks)
!kaggle datasets download -d bitsandlayers/sar-oil-spill-segmentation-dataset-sos -p ml/data --unzip
DATA_ROOT = 'ml/data/dataset'   # {train,test}/{sentinel,palsar}/{image,label}; loader prefers sentinel
import os; print('train sentinel imgs:', len(os.listdir(f'{DATA_ROOT}/train/sentinel/image')))

In [ ]:
# 4. Train all five (identical loader/loss/tiles). ~40 epochs on a T4.
#    Quick check first? add  --limit 400 --epochs 3.
!python -m ml.train --data-root "$DATA_ROOT" --epochs 40 --batch-size 16

In [ ]:
# 5. Comparison table + auto-recommended winner
import json
m = json.load(open('ml/metrics.json'))
print('recommended:', m['recommended'], '| test images:', m['test_images'])
for r in m['models']:
    print(f"{r['name']:<22} mIoU={r['miou']:.3f}  oil={r['iou'].get('oil')}  {r['params_m']}M  {r['cpu_ms']}ms")

In [ ]:
# 6. Export the winner to ONNX + sidecar
WINNER = m['recommended']
!python -m ml.export --model-name $WINNER --weights ml/weights/$WINNER.pt

In [ ]:
# 7. Download the artefacts to commit into the API (ml/weights/ is git-ignored except .gitkeep)
from google.colab import files
files.download(f'ml/weights/{WINNER}.onnx')
files.download(f'ml/weights/{WINNER}.json')
files.download('ml/metrics.json')
print('Place the .onnx + .json in ml/weights/, commit metrics.json. /api/health flips to engine=unet.')